In [1]:
import os
from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv())

In [2]:
os.environ["GROQ_API_KEY"] = "Your Groq API"

In [3]:
!pip install langchain-groq

Defaulting to user installation because normal site-packages is not writeable


## Completion model

In [4]:
from langchain_groq import ChatGroq
llmModel = ChatGroq(
    model="llama-3.1-8b-instant" 
)

# chat completion model

In [5]:
chatModel = llmModel

## Prompts and Prompt Templates

In [6]:
# This is for completions model
from langchain_core.prompts import PromptTemplate

prompt_template = PromptTemplate.from_template(
    "Tell me a {adjective} story about {topic}"
)

llmModelPrompt = prompt_template.format(
    adjective="curious",
    topic="Prophet Muhammad (SAW)"
)

In [7]:
response = llmModel.invoke(llmModelPrompt)
print(response.content)

One of the most fascinating stories about Prophet Muhammad (SAW) is the story of the 'Miracle of Muddaththir' or 'The Prophet's Night Journey'.

It is said that one night, about two years after the Prophet's migration to Medina, he was awakened from his sleep for a mysterious journey to Jerusalem. This night journey, known as 'Isra' and 'M'iraj', is considered one of the most significant events in Islamic history.

As the story goes, the Prophet Muhammad (SAW) was taken by the angel Gabriel to the Temple Mount in Jerusalem, where he met other prophets, including Abraham, Moses, and Jesus. The other prophets asked the Prophet to lead them in prayer, but he politely declined, saying that he was not worthy of leading them.

After meeting the prophets, the Prophet Muhammad (SAW) was then taken on a journey to the heavens, where he encountered various levels of heaven, including the Garden of Eden. At the first heaven, he met the Prophet Idris (Enoch), who was one of the earliest prophets.


In [8]:
# chat completion model

from langchain_core.prompts import ChatPromptTemplate

chat_template = ChatPromptTemplate.from_messages(
    [
        ("system", "You are an {profession} expert on {topic}."),
        ("human", "Hello, Mr. {profession}, can you please answer a question?"),
        ("ai", "Sure!"),
        ("human", "{user_input}"),
    ]
)

messages = chat_template.format_messages(
    profession="Historian",
    topic="The life of Prophet Muhammad (SAW)",
    user_input="How many children did Prophet Muhammad (SAW) have?"
)

response = chatModel.invoke(messages)
print(response.content)


Prophet Muhammad (SAW) had a total of 9 children from his marriages. 

1. Al-Qasim (son with his first wife Khadijah) - He passed away at a young age.
2. Zainab (daughter with Khadijah) - She passed away before the migration to Medina.
3. Ruqayyah (daughter with Khadijah) - She passed away in Medina.
4. Umm Kulthum (daughter with Khadijah) - She passed away before the Battle of Uhud.
5. Fatimah (daughter with Khadijah) - She was the only child of Prophet Muhammad (SAW) to have children.
6. Abdullah (son with his second wife Aisha's sister, Asma's daughter Umamah) was adopted by him but he did not have biological children with her 
7. Abd-Allah (son with his wife Fatimah bint Hizam) - He passed away at a young age.
8. Umar (son with his wife Jamila) - He passed away at a young age.
9. Ibrahim (son with his wife Maria al-Qibtiyya) - He passed away at a young age.

Note: Some sources may list the children differently or include some of the Prophet's other children who passed away in infan

# Few Shot Prompting

In [9]:
from langchain_core.prompts import FewShotChatMessagePromptTemplate

examples = [
    {"input": "hi!", "output": "Assalamualikum"},
    {"input": "bye!", "output": "Fi amanillah"},
]

example_prompt = ChatPromptTemplate.from_messages(
    [
        ("human", "{input}"),
        ("ai", "{output}"),
    ]
)

few_shot_prompt = FewShotChatMessagePromptTemplate(
    example_prompt=example_prompt,
    examples=examples,
)

final_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "You are an English-Arabic translator."),
        few_shot_prompt,
        ("human", "{input}"),
    ]
)

# Chains

In [10]:
from langchain_core.prompts import FewShotChatMessagePromptTemplate

examples = [
    {"input": "hi!", "output": "Assalamualikum"},
    {"input": "bye!", "output": "Fi amanillah"},
]

example_prompt = ChatPromptTemplate.from_messages(
    [
        ("human", "{input}"),
        ("ai", "{output}"),
    ]
)

few_shot_prompt = FewShotChatMessagePromptTemplate(
    example_prompt=example_prompt,
    examples=examples,
)

final_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "You are an English-Arabic translator."),
        few_shot_prompt,
        ("human", "{input}"),
    ]
)


chain = final_prompt | chatModel

res = chain.invoke({"input": "How are you?"})
print(res.content)

Mafish shay, shukraan. (I'm fine, thank you)


In [11]:
res

AIMessage(content="Mafish shay, shukraan. (I'm fine, thank you)", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 20, 'prompt_tokens': 82, 'total_tokens': 102, 'completion_time': 0.046131495, 'completion_tokens_details': None, 'prompt_time': 0.017412839, 'prompt_tokens_details': None, 'queue_time': 0.045988361, 'total_time': 0.063544334}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_f757f4b0bf', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019d4930-bfd6-7f40-9753-3838342910b5-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 82, 'output_tokens': 20, 'total_tokens': 102})

# Output Parsers

In [12]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import JsonOutputParser

In [13]:
json_prompt = PromptTemplate.from_template(
    "Return a JSON object with an `answer` key that answers the following question: {question}"
)

json_parser = JsonOutputParser()

json_chain = json_prompt | llmModel | json_parser

In [14]:
json_parser.get_format_instructions()

'Return a JSON object.'

In [19]:
res  = json_chain.invoke({"question": "What is the biggest City?"})

In [20]:
res

{'answer': {'text': 'The biggest city is Tokyo, Japan, with a population of over 38 million people in its metropolitan area.',
  'source': 'According to the United Nations, Tokyo is considered the largest city in the world by population, with a population density of over 6,000 people per square kilometer.'}}

In [21]:
type(res)

dict

#### Optionally, you can use Pydantic to define a custom output format

In [18]:
from langchain_core.output_parsers import JsonOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_core.pydantic_v1 import BaseModel, Field

ModuleNotFoundError: No module named 'langchain_core.pydantic_v1'

In [ ]:
# Define a Pydantic Object with the desired output format.
class Joke(BaseModel):
    setup: str = Field(description="question to set up a joke")
    punchline: str = Field(description="answer to resolve the joke")

In [ ]:
# Define the parser referring the Pydantic Object
parser = JsonOutputParser(pydantic_object=Joke)

# Add the parser format instructions in the prompt definition.
prompt = PromptTemplate(
    template="Answer the user query.\n{format_instructions}\n{query}\n",
    input_variables=["query"],
    partial_variables={"format_instructions": parser.get_format_instructions()},
)

# Create a chain with the prompt and the parser
chain = prompt | chatModel | parser

chain.invoke({"query": "Tell me a joke."})